In [39]:
import numpy as np
import pandas as pd
import re
from collections import Counter
import math
import random
from scipy.sparse import csr_matrix, lil_matrix

In [40]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("="*80)
print("SENTIMENT ANALYSIS ON IMDB MOVIE REVIEWS")
print("="*80)

SENTIMENT ANALYSIS ON IMDB MOVIE REVIEWS


In [41]:
# ============================================
# STEP 1: LOAD DATASET
# ============================================
print("\n" + "="*80)
print("STEP 1: Loading Dataset")
print("="*80)

df = pd.read_csv("IMDB Dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())


STEP 1: Loading Dataset
Dataset shape: (50000, 2)

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [42]:
# Convert sentiment to numeric
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [43]:
# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [44]:
print("\nFirst 3 reviews:")
for i in range(3):
    print(f"\nReview {i+1} (sentiment: {'positive' if df['sentiment'].iloc[i]==1 else 'negative'}):")
    print(df['review'].iloc[i][:200] + "...")


First 3 reviews:

Review 1 (sentiment: positive):
I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ev...

Review 2 (sentiment: positive):
Not many television shows appeal to quite as many different kinds of fans like Farscape does...I know youngsters and 30/40+ years old;fans both Male and Female in as many different countries as you ca...

Review 3 (sentiment: negative):
The film quickly gets to a major chase scene with ever increasing destruction. The first really bad thing is the guy hijacking Steven Seagal would have been beaten to pulp by Seagal's driving, but tha...


In [45]:
# ============================================
# STEP 2: PREPROCESSING
# ============================================
print("\n" + "="*80)
print("STEP 2: Preprocessing")
print("="*80)


STEP 2: Preprocessing


In [46]:
class TextPreprocessor:
    """
    Complete text preprocessing pipeline including:
    - Tokenization
    - Normalization (lowercasing)
    - Punctuation handling (with trade-off discussion)
    - Number handling (with trade-off discussion)
    - Stopword handling (keeping negations)
    - Stemming
    - Emoji/Hashtag handling
    """
    
    def __init__(self):
        # Stopwords (excluding negations which are crucial for sentiment)
        self.stopwords = set([
            'a', 'about', 'above', 'after', 'again', 'all', 'am', 'an', 'and',
            'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before',
            'being', 'below', 'both', 'but', 'by', 'can', 'did', 'do', 'does',
            'doing', 'down', 'during', 'each', 'few', 'for', 'from', 'further',
            'had', 'has', 'have', 'having', 'he', 'her', 'here', 'hers', 'herself',
            'him', 'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'it',
            'its', 'itself', 'just', 'me', 'more', 'most', 'my', 'myself',
            'nor',  # 'nor' is negation, but we keep it
            'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours',
            'ourselves', 'out', 'over', 'own', 'same', 'she', 'should', 'so',
            'some', 'such', 'than', 'that', 'the', 'their', 'theirs', 'them',
            'themselves', 'then', 'there', 'these', 'they', 'this', 'those',
            'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was',
            'we', 'were', 'what', 'when', 'where', 'which', 'while', 'who',
            'whom', 'why', 'will', 'with', 'you', 'your', 'yours', 'yourself',
            'yourselves'
        ])
        
        # Negation words to KEEP (they're crucial for sentiment)
        self.negations = {'not', 'no', 'never', 'nothing', 'nowhere', 'none',
                          'nobody', 'neither', 'nor', "isn't", "aren't", "wasn't",
                          "weren't", "haven't", "hasn't", "hadn't", "won't",
                          "wouldn't", "don't", "doesn't", "didn't", "can't",
                          "couldn't", "shouldn't", "mightn't", "mustn't"}
        
        # Emoji mapping
        self.emoji_dict = {
            '😀': ' smile ', '😃': ' smile ', '😄': ' smile ', '😁': ' smile ',
            '😆': ' laugh ', '😂': ' laugh ', '🤣': ' laugh ', '😊': ' smile ',
            '😇': ' smile ', '🙂': ' smile ', '😉': ' wink ', '😌': ' relieved ',
            '😍': ' love ', '🥰': ' love ', '😘': ' love ', '😗': ' kiss ',
            '😙': ' kiss ', '😚': ' kiss ', '😋': ' yummy ', '😛': ' playful ',
            '😝': ' playful ', '😜': ' playful ', '🤪': ' crazy ', '🤨': ' suspicious ',
            '🧐': ' curious ', '😒': ' unimpressed ', '😔': ' sad ', '😕': ' confused ',
            '🙁': ' sad ', '☹️': ' sad ', '😣': ' struggling ', '😖': ' frustrated ',
            '😫': ' tired ', '😩': ' exhausted ', '🥺': ' pleading ', '😢': ' sad ',
            '😭': ' cry ', '😤': ' angry ', '😠': ' angry ', '😡': ' angry ',
            '🤬': ' angry ', '🤯': ' shocked ', '😳': ' embarrassed ', '🥵': ' hot ',
            '🥶': ' cold ', '😱': ' scared ', '😨': ' scared ', '😰': ' scared ',
            '😥': ' sad ', '😓': ' sad ', '🤗': ' hug ', '🤔': ' thinking ',
            '🤭': ' surprised ', '🤫': ' quiet ', '🤥': ' liar ', '😶': ' speechless ',
            '😐': ' neutral ', '😑': ' neutral ', '😬': ' awkward ', '🙄': ' rolling eyes ',
            '😯': ' surprised ', '😦': ' sad ', '😧': ' anguished ', '😮': ' surprised ',
            '😲': ' shocked ', '😴': ' sleepy ', '🤤': ' drooling ', '😪': ' sleepy ',
            '😵': ' dizzy ', '🤐': ' zipped ', '🥴': ' woozy ', '🤢': ' sick ',
            '🤮': ' vomit ', '🤧': ' sneeze ', '😷': ' sick ', '🤒': ' sick ',
            '🤕': ' hurt ', '🤑': ' money ', '🤠': ' cowboy ', '😎': ' cool ',
            '🤓': ' nerd ', '🧐': ' curious ', '😕': ' confused ', '🙁': ' sad ',
            '☹️': ' sad ', '😮': ' surprised ', '😯': ' surprised ', '😲': ' shocked ',
            '😳': ' embarrassed ', '🥺': ' pleading ', '😦': ' sad ', '😧': ' anguished ',
            '😨': ' scared ', '😰': ' scared ', '😥': ' sad ', '😢': ' sad ',
            '😭': ' cry ', '😱': ' scared ', '😖': ' frustrated ', '😣': ' struggling ',
            '😞': ' disappointed ', '😓': ' sad ', '😩': ' exhausted ', '😫': ' tired ',
            '🥱': ' yawn ', '😤': ' angry ', '😡': ' angry ', '😠': ' angry ',
            '🤬': ' angry ', '😈': ' devil ', '👿': ' devil ', '💀': ' death ',
            '☠️': ' death ', '💩': ' poop ', '🤡': ' clown ', '👹': ' monster ',
            '👺': ' monster ', '👻': ' ghost ', '👽': ' alien ', '👾': ' alien ',
            '🤖': ' robot ', '🎃': ' halloween ', '😺': ' cat ', '😸': ' cat ',
            '😹': ' cat ', '😻': ' cat ', '😼': ' cat ', '😽': ' cat ',
            '🙀': ' cat ', '😿': ' cat ', '😾': ' cat '
        }
        
        # Common contractions to expand
        self.contractions = {
            "ain't": "is not", "aren't": "are not", "can't": "cannot",
            "could've": "could have", "couldn't": "could not", "didn't": "did not",
            "doesn't": "does not", "don't": "do not", "hadn't": "had not",
            "hasn't": "has not", "haven't": "have not", "he'd": "he would",
            "he'll": "he will", "he's": "he is", "how'd": "how did",
            "how'll": "how will", "how's": "how is", "i'd": "i would",
            "i'll": "i will", "i'm": "i am", "i've": "i have",
            "isn't": "is not", "it'd": "it would", "it'll": "it will",
            "it's": "it is", "let's": "let us", "might've": "might have",
            "mightn't": "might not", "must've": "must have", "mustn't": "must not",
            "needn't": "need not", "oughtn't": "ought not", "shan't": "shall not",
            "she'd": "she would", "she'll": "she will", "she's": "she is",
            "should've": "should have", "shouldn't": "should not", "that's": "that is",
            "there's": "there is", "they'd": "they would", "they'll": "they will",
            "they're": "they are", "they've": "they have", "wasn't": "was not",
            "we'd": "we would", "we'll": "we will", "we're": "we are",
            "we've": "we have", "weren't": "were not", "what'll": "what will",
            "what're": "what are", "what's": "what is", "what've": "what have",
            "where'd": "where did", "where's": "where is", "who'd": "who would",
            "who'll": "who will", "who's": "who is", "who've": "who have",
            "why'd": "why did", "why's": "why is", "won't": "will not",
            "would've": "would have", "wouldn't": "would not", "you'd": "you would",
            "you'll": "you will", "you're": "you are", "you've": "you have"
        }
    
    def expand_contractions(self, text):
        """Expand common contractions"""
        for contraction, expansion in self.contractions.items():
            text = text.replace(contraction, expansion)
            text = text.replace(contraction.capitalize(), expansion.capitalize())
        return text
    
    def handle_emojis(self, text):
        """Replace emojis with their text meanings"""
        for emoji, meaning in self.emoji_dict.items():
            if emoji in text:
                text = text.replace(emoji, meaning)
        return text
    
    def handle_hashtags(self, text):
        """Handle hashtags - remove # but keep the word"""
        # Find hashtags
        hashtags = re.findall(r'#(\w+)', text)
        for tag in hashtags:
            # Keep the word but remove #
            text = text.replace('#' + tag, ' ' + tag + ' ')
        return text
    
    def remove_html(self, text):
        """Remove HTML tags"""
        return re.sub(r'<.*?>', '', text)
    
    def remove_urls(self, text):
        """Remove URLs"""
        url_pattern = r'https?://\S+|www\.\S+'
        return re.sub(url_pattern, ' ', text)
    
    def normalize(self, text):
        """Normalization: lowercasing and basic cleaning"""
        # Convert to lowercase
        text = text.lower()
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        
        return text
    
    def discuss_punctuation_tradeoff(self):
        """Discussion of punctuation removal trade-offs (required by image)"""
        discussion = """
        TRADE-OFFS IN PUNCTUATION HANDLING:
        
        Option 1: Remove all punctuation
        Pros:
        - Reduces vocabulary size
        - Treats "good" and "good!" as the same word
        - Simplifies the model
        
        Cons:
        - Loses emotional markers (!!! indicates excitement)
        - Loses sentence structure information
        - Can't distinguish between "good?" and "good!"
        
        Option 2: Keep punctuation
        Pros:
        - Preserves emotional intensity (!!!, ???)
        - Can capture specific patterns like "?!"
        
        Cons:
        - Increases vocabulary size
        - Creates sparse features
        - May overfit to specific punctuation patterns
        
        Option 3: Selective handling (convert to markers)
        - Replace ! with <exclamation>
        - Replace ? with <question>
        - This balances both approaches
        
        For sentiment analysis, punctuation often carries emotional weight,
        so complete removal might lose signal. However, for simplicity and
        to reduce sparsity, we'll remove punctuation but acknowledge this trade-off.
        """
        print(discussion)
    
    def discuss_numbers_tradeoff(self):
        """Discussion of number removal trade-offs (required by image)"""
        discussion = """
        TRADE-OFFS IN NUMBER HANDLING:
        
        Option 1: Remove all numbers
        Pros:
        - Reduces vocabulary size significantly
        - Prevents overfitting to specific numbers (e.g., "10/10")
        
        Cons:
        - Loses rating information (e.g., "9/10" vs "2/10")
        - Loses year references that might indicate film era preferences
        
        Option 2: Keep numbers
        Pros:
        - Captures explicit ratings
        - Can identify decade-specific references
        
        Cons:
        - Creates many rare features (each number becomes a feature)
        - Numbers like "10" might be too specific
        
        Option 3: Normalize numbers (replace with <num>)
        - Treats all numbers as the same token
        - Preserves that a number was present without overfitting
        
        For movie reviews, explicit ratings like "8/10" are strong signals,
        so removal might hurt performance. But to reduce vocabulary size,
        we'll replace numbers with a special <num> token.
        """
        print(discussion)
    
    def handle_numbers(self, text):
        """Replace numbers with <num> token"""
        # Replace standalone numbers with <num>
        text = re.sub(r'\b\d+\b', ' <num> ', text)
        # Replace fractions like 8/10
        text = re.sub(r'\b\d+/\d+\b', ' <rating> ', text)
        return text
    
    def simple_stem(self, word):
        """Manual stemming implementation"""
        # Common suffixes to remove
        suffixes = ['ing', 'ed', 'ly', 'es', 's', 'ment', 'tion', 'able']
        
        # Don't stem very short words
        if len(word) <= 3:
            return word
        
        # Special cases
        if word.endswith('ing') and len(word) > 5:
            # Check if it's a common word
            if word in ['something', 'anything', 'everything', 'morning']:
                return word
            # Check if it's a verb in -ing form
            if word[-4:-3] == word[-4:-3].lower():
                return word[:-3]
        elif word.endswith('ed') and len(word) > 4:
            # Check for irregular past tense
            if word in ['said', 'made', 'went', 'came', 'took']:
                return word
            # Keep the base for regular past tense
            if word[-4:-3] == word[-4:-3].lower():
                return word[:-2]
        elif word.endswith('ly') and len(word) > 4:
            return word[:-2]
        elif word.endswith('es') and len(word) > 4:
            return word[:-2]
        elif word.endswith('s') and len(word) > 3:
            # Don't remove 's' from words like 'this', 'thus', 'yes'
            if word not in ['this', 'thus', 'yes', 'was', 'has']:
                return word[:-1]
        
        return word
    
    def stem_tokens(self, tokens):
        """Apply stemming to a list of tokens"""
        return [self.simple_stem(token) for token in tokens]
    
    def remove_stopwords(self, tokens):
        """Remove stopwords but keep negations"""
        return [token for token in tokens 
                if token not in self.stopwords or token in self.negations]
    
    def preprocess(self, text):
        """
        Complete preprocessing pipeline with proper ordering
        """
        # Step 1: Handle special elements first
        text = self.handle_emojis(text)
        text = self.handle_hashtags(text)
        text = self.expand_contractions(text)
        
        # Step 2: Clean markup
        text = self.remove_html(text)
        text = self.remove_urls(text)
        
        # Step 3: Normalize
        text = self.normalize(text)
        
        # Step 4: Handle numbers (with trade-off consideration)
        text = self.handle_numbers(text)
        
        # Step 5: Remove punctuation
        # Discuss trade-off (as required by image)
        # self.discuss_punctuation_tradeoff()  # Uncomment to see discussion
        text = re.sub(r'[^\w\s<>]', ' ', text)  # Keep < and > for special tokens
        
        # Step 6: Tokenize
        tokens = text.split()
        
        # Step 7: Remove stopwords (keeping negations)
        tokens = self.remove_stopwords(tokens)
        
        # Step 8: Apply stemming
        tokens = self.stem_tokens(tokens)
        
        return tokens

# Initialize preprocessor
preprocessor = TextPreprocessor()

In [47]:
# Apply preprocessing to all reviews
print("\nPreprocessing 50,000 reviews...")
df['tokens'] = df['review'].apply(preprocessor.preprocess)


Preprocessing 50,000 reviews...


In [48]:
# Show example
print("\nExample preprocessing:")
print("Original (first 200 chars):")
print(df['review'].iloc[0][:200] + "...")
print("\nAfter preprocessing (first 30 tokens):")
print(df['tokens'].iloc[0][:30])


Example preprocessing:
Original (first 200 chars):
I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ev...

After preprocessing (first 30 tokens):
['real', 'lik', 'summerslam', 'due', 'look', 'arena', 'curtain', 'look', 'overall', 'interest', 'reason', 'anyway', 'could', 'one', 'best', 'summerslam', 's', 'ever', 'wwf', 'not', 'lex', 'luger', 'main', 'event', 'against', 'yokozuna', 'now', 'time', 'ok', 'huge']


In [49]:
# Show token length statistics
token_lengths = df['tokens'].apply(len)
print(f"\nToken statistics:")
print(f"  Average tokens per review: {token_lengths.mean():.1f}")
print(f"  Min tokens: {token_lengths.min()}")
print(f"  Max tokens: {token_lengths.max()}")


Token statistics:
  Average tokens per review: 124.7
  Min tokens: 3
  Max tokens: 1470


In [50]:
# ============================================
# STEP 3: TRAIN/VAL/TEST SPLIT
# ============================================
print("\n" + "="*80)
print("STEP 3: Train/Validation/Test Split")
print("="*80)


STEP 3: Train/Validation/Test Split


In [51]:
# 70% train, 10% validation, 20% test
n = len(df)
train_end = int(0.7 * n)
val_end = int(0.8 * n)

train_df = df[:train_end].copy()
val_df = df[train_end:val_end].copy()
test_df = df[val_end:].copy()

print(f"Training set: {len(train_df)} reviews ({len(train_df)/n*100:.1f}%)")
print(f"Validation set: {len(val_df)} reviews ({len(val_df)/n*100:.1f}%)")
print(f"Test set: {len(test_df)} reviews ({len(test_df)/n*100:.1f}%)")

Training set: 35000 reviews (70.0%)
Validation set: 5000 reviews (10.0%)
Test set: 10000 reviews (20.0%)


In [52]:
print(f"\nTraining set sentiment distribution:")
print(f"  Positive: {train_df['sentiment'].sum()} ({train_df['sentiment'].sum()/len(train_df)*100:.1f}%)")
print(f"  Negative: {len(train_df) - train_df['sentiment'].sum()} ({(len(train_df) - train_df['sentiment'].sum())/len(train_df)*100:.1f}%)")


Training set sentiment distribution:
  Positive: 17483 (50.0%)
  Negative: 17517 (50.0%)


In [53]:
# ============================================
# STEP 4: FEATURE EXTRACTION (TF-IDF)
# ============================================
print("\n" + "="*80)
print("STEP 4: Feature Extraction (TF-IDF)")
print("="*80)


STEP 4: Feature Extraction (TF-IDF)


In [62]:
def build_vocabulary(tokenized_texts, max_features=5000, min_freq=5):
    """
    Build vocabulary with size limit to manage memory
    """
    # Count word frequencies
    word_counter = Counter()
    for tokens in tokenized_texts:
        word_counter.update(tokens)
    
    # Filter by minimum frequency
    filtered_words = {word: count for word, count in word_counter.items() 
                     if count >= min_freq}
    
    # Get most common words (limited to max_features)
    most_common = Counter(filtered_words).most_common(max_features)
    
    # Create vocabulary
    vocab = {word: idx for idx, (word, _) in enumerate(most_common)}
    
    print(f"Vocabulary size: {len(vocab)} words")
    print(f"  Total unique words: {len(word_counter)}")
    print(f"  Words with frequency >= {min_freq}: {len(filtered_words)}")
    
    return vocab, word_counter

In [64]:
# Build vocabulary from training data only (limit features to save memory)
vocab, word_counter = build_vocabulary(train_df['tokens'], max_features=5000, min_freq=5)

Vocabulary size: 5000 words
  Total unique words: 72041
  Words with frequency >= 5: 26921


In [65]:
# Create reverse mapping
index_to_word = {idx: word for word, idx in vocab.items()}

In [66]:
def compute_idf_sparse(tokenized_texts, vocab):
    """
    Compute IDF using sparse representation
    """
    N = len(tokenized_texts)
    doc_freq = np.zeros(len(vocab))
    
    for tokens in tokenized_texts:
        # Get unique words in document
        unique_words = set(tokens)
        for word in unique_words:
            if word in vocab:
                doc_freq[vocab[word]] += 1
    
    # Compute IDF with smoothing
    idf = np.log((N + 1) / (doc_freq + 1)) + 1
    
    return idf

# Compute IDF from training data
idf = compute_idf_sparse(train_df['tokens'], vocab)

In [67]:
def create_sparse_tfidf_matrix(tokenized_texts, vocab, idf):
    """
    Create sparse TF-IDF matrix using lil_matrix for efficient construction
    """
    n_docs = len(tokenized_texts)
    n_features = len(vocab)
    
    # Use LIL matrix for efficient incremental building
    lil = lil_matrix((n_docs, n_features), dtype=np.float32)
    
    for doc_idx, tokens in enumerate(tokenized_texts):
        if doc_idx % 1000 == 0:
            print(f"  Processing document {doc_idx}/{n_docs}")
        
        # Count term frequencies
        term_counts = {}
        for word in tokens:
            if word in vocab:
                term_counts[word] = term_counts.get(word, 0) + 1
        
        # Compute TF-IDF
        doc_length = len(tokens)
        if doc_length > 0:
            for word, count in term_counts.items():
                tf = count / doc_length
                word_idx = vocab[word]
                lil[doc_idx, word_idx] = tf * idf[word_idx]
    
    # Convert to CSR format for efficient operations
    return lil.tocsr()

In [68]:
print("\nCreating sparse TF-IDF matrices...")
X_train = create_sparse_tfidf_matrix(train_df['tokens'], vocab, idf)
X_val = create_sparse_tfidf_matrix(val_df['tokens'], vocab, idf)
X_test = create_sparse_tfidf_matrix(test_df['tokens'], vocab, idf)


Creating sparse TF-IDF matrices...
  Processing document 0/35000
  Processing document 1000/35000
  Processing document 2000/35000
  Processing document 3000/35000
  Processing document 4000/35000
  Processing document 5000/35000
  Processing document 6000/35000
  Processing document 7000/35000
  Processing document 8000/35000
  Processing document 9000/35000
  Processing document 10000/35000
  Processing document 11000/35000
  Processing document 12000/35000
  Processing document 13000/35000
  Processing document 14000/35000
  Processing document 15000/35000
  Processing document 16000/35000
  Processing document 17000/35000
  Processing document 18000/35000
  Processing document 19000/35000
  Processing document 20000/35000
  Processing document 21000/35000
  Processing document 22000/35000
  Processing document 23000/35000
  Processing document 24000/35000
  Processing document 25000/35000
  Processing document 26000/35000
  Processing document 27000/35000
  Processing document 280

In [69]:
y_train = train_df['sentiment'].values.astype(np.float32)
y_val = val_df['sentiment'].values.astype(np.float32)
y_test = test_df['sentiment'].values.astype(np.float32)

In [70]:
print(f"\nTraining TF-IDF shape: {X_train.shape}")
print(f"Training matrix density: {X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}")
print(f"Non-zero elements: {X_train.nnz}")


Training TF-IDF shape: (35000, 5000)
Training matrix density: 1.6873%
Non-zero elements: 2952757


In [72]:
# ============================================
# STEP 5: LOGISTIC REGRESSION WITH SPARSE MATRICES
# ============================================
print("\n" + "="*80)
print("STEP 5: Logistic Regression with Sparse Matrices")
print("="*80)


STEP 5: Logistic Regression with Sparse Matrices


In [73]:

class SparseLogisticRegression:
    """
    Memory-efficient Logistic Regression for sparse matrices
    """
    
    def __init__(self, learning_rate=0.1, epochs=50, reg_lambda=0.01, batch_size=256, verbose=True):
        self.lr = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.batch_size = batch_size
        self.verbose = verbose
        self.weights = None
        self.bias = 0
        self.loss_history = []
    
    def sigmoid(self, z):
        """Numerically stable sigmoid"""
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss with L2 regularization"""
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        # Cross-entropy loss
        ce_loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        
        # L2 regularization
        if self.reg_lambda > 0:
            reg_loss = (self.reg_lambda / (2 * len(y_true))) * np.sum(self.weights**2)
        else:
            reg_loss = 0
        
        return ce_loss + reg_loss
    
    def fit(self, X, y, X_val=None, y_val=None):
        """
        Train using mini-batch gradient descent
        """
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features, dtype=np.float32)
        self.bias = 0.0
        
        # For tracking best model
        best_val_acc = 0
        best_weights = None
        best_bias = None
        
        for epoch in range(self.epochs):
            # Mini-batch training
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            epoch_losses = []
            
            for i in range(0, n_samples, self.batch_size):
                end = min(i + self.batch_size, n_samples)
                X_batch = X_shuffled[i:end]
                y_batch = y_shuffled[i:end]
                
                # Forward pass (sparse dot product)
                z = X_batch.dot(self.weights) + self.bias
                y_pred = self.sigmoid(z)
                
                # Compute loss
                loss = self.compute_loss(y_batch, y_pred)
                epoch_losses.append(loss)
                
                # Compute gradients
                error = y_pred - y_batch
                
                # Gradient for weights (sparse)
                dw = (X_batch.T.dot(error) / len(y_batch)) + (self.reg_lambda * self.weights / len(y_batch))
                db = np.mean(error)
                
                # Update parameters
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
            
            # Record average loss
            avg_loss = np.mean(epoch_losses)
            self.loss_history.append(avg_loss)
            
            # Validation
            if X_val is not None and y_val is not None:
                val_acc = self.score(X_val, y_val)
                
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_weights = self.weights.copy()
                    best_bias = self.bias
            
            # Print progress
            if self.verbose and (epoch + 1) % 5 == 0:
                train_acc = self.score(X, y)
                print(f"Epoch {epoch+1}/{self.epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        # Restore best weights
        if best_weights is not None:
            self.weights = best_weights
            self.bias = best_bias
        
        return self
    
    def predict_proba(self, X):
        """Predict probabilities"""
        z = X.dot(self.weights) + self.bias
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predict class labels"""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """Calculate accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

In [74]:
# Hyperparameter tuning (simplified)
print("\nTraining Logistic Regression...")
print("-" * 40)


Training Logistic Regression...
----------------------------------------


In [75]:
# Try a few parameter combinations
param_combinations = [
    (0.1, 0.001),
    (0.1, 0.01),
    (0.05, 0.001),
    (0.05, 0.01)
]

best_val_acc = 0
best_params = None
best_model = None

for lr, reg in param_combinations:
    print(f"\nTesting lr={lr}, reg={reg}")
    
    model = SparseLogisticRegression(
        learning_rate=lr,
        epochs=10,
        reg_lambda=reg,
        batch_size=256,
        verbose=False
    )
    
    model.fit(X_train, y_train, X_val, y_val)
    val_acc = model.score(X_val, y_val)
    
    print(f"  Validation Accuracy: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = (lr, reg)
        best_model = model

print("\n" + "="*40)
print(f"Best parameters: lr={best_params[0]}, reg={best_params[1]}")
print(f"Best validation accuracy: {best_val_acc:.4f}")


Testing lr=0.1, reg=0.001
  Validation Accuracy: 0.8184

Testing lr=0.1, reg=0.01
  Validation Accuracy: 0.8166

Testing lr=0.05, reg=0.001
  Validation Accuracy: 0.8194

Testing lr=0.05, reg=0.01
  Validation Accuracy: 0.8196

Best parameters: lr=0.05, reg=0.01
Best validation accuracy: 0.8196


In [76]:
# Train final model with best parameters
print("\nTraining final model...")
final_model = SparseLogisticRegression(
    learning_rate=best_params[0],
    epochs=50,
    reg_lambda=best_params[1],
    batch_size=256,
    verbose=True
)

final_model.fit(X_train, y_train, X_val, y_val)


Training final model...
Epoch 5/50 | Loss: 0.6901 | Train Acc: 0.8189 | Val Acc: 0.8108
Epoch 10/50 | Loss: 0.6868 | Train Acc: 0.8220 | Val Acc: 0.8172
Epoch 15/50 | Loss: 0.6835 | Train Acc: 0.8242 | Val Acc: 0.8204
Epoch 20/50 | Loss: 0.6803 | Train Acc: 0.8247 | Val Acc: 0.8210
Epoch 25/50 | Loss: 0.6772 | Train Acc: 0.8205 | Val Acc: 0.8124
Epoch 30/50 | Loss: 0.6741 | Train Acc: 0.8227 | Val Acc: 0.8158
Epoch 35/50 | Loss: 0.6711 | Train Acc: 0.8210 | Val Acc: 0.8130
Epoch 40/50 | Loss: 0.6681 | Train Acc: 0.8257 | Val Acc: 0.8218
Epoch 45/50 | Loss: 0.6652 | Train Acc: 0.8236 | Val Acc: 0.8162
Epoch 50/50 | Loss: 0.6623 | Train Acc: 0.8203 | Val Acc: 0.8134


In [77]:
# ============================================
# STEP 6: EVALUATION
# ============================================
print("\n" + "="*80)
print("STEP 6: Model Evaluation")
print("="*80)


STEP 6: Model Evaluation


In [78]:
def evaluate_model(model, X, y, dataset_name=""):
    """Comprehensive evaluation"""
    y_pred = model.predict(X)
    
    # Confusion matrix
    tp = np.sum((y == 1) & (y_pred == 1))
    tn = np.sum((y == 0) & (y_pred == 0))
    fp = np.sum((y == 0) & (y_pred == 1))
    fn = np.sum((y == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / len(y)
    
    # Metrics for positive class
    precision_pos = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_pos = 2 * (precision_pos * recall_pos) / (precision_pos + recall_pos) if (precision_pos + recall_pos) > 0 else 0
    
    # Metrics for negative class
    precision_neg = tn / (tn + fn) if (tn + fn) > 0 else 0
    recall_neg = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_neg = 2 * (precision_neg * recall_neg) / (precision_neg + recall_neg) if (precision_neg + recall_neg) > 0 else 0
    
    print(f"\n{dataset_name} Results:")
    print("-" * 40)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"              Predicted")
    print(f"              Pos    Neg")
    print(f"Actual Pos    {tp:5d}  {fn:5d}")
    print(f"Actual Neg    {fp:5d}  {tn:5d}")
    print(f"\nPositive Class (1): Precision={precision_pos:.4f}, Recall={recall_pos:.4f}, F1={f1_pos:.4f}")
    print(f"Negative Class (0): Precision={precision_neg:.4f}, Recall={recall_neg:.4f}, F1={f1_neg:.4f}")
    
    return {
        'accuracy': accuracy,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'precision_pos': precision_pos, 'recall_pos': recall_pos, 'f1_pos': f1_pos,
        'precision_neg': precision_neg, 'recall_neg': recall_neg, 'f1_neg': f1_neg
    }

In [79]:
# Evaluate
train_results = evaluate_model(final_model, X_train, y_train, "TRAINING SET")
val_results = evaluate_model(final_model, X_val, y_val, "VALIDATION SET")
test_results = evaluate_model(final_model, X_test, y_test, "TEST SET")

# Error analysis
print("\n" + "="*80)
print("STEP 7: Error Analysis")
print("="*80)


TRAINING SET Results:
----------------------------------------
Accuracy: 0.8257

Confusion Matrix:
              Predicted
              Pos    Neg
Actual Pos    14576   2907
Actual Neg     3195  14322

Positive Class (1): Precision=0.8202, Recall=0.8337, F1=0.8269
Negative Class (0): Precision=0.8313, Recall=0.8176, F1=0.8244

VALIDATION SET Results:
----------------------------------------
Accuracy: 0.8218

Confusion Matrix:
              Predicted
              Pos    Neg
Actual Pos     2080    441
Actual Neg      450   2029

Positive Class (1): Precision=0.8221, Recall=0.8251, F1=0.8236
Negative Class (0): Precision=0.8215, Recall=0.8185, F1=0.8200

TEST SET Results:
----------------------------------------
Accuracy: 0.8235

Confusion Matrix:
              Predicted
              Pos    Neg
Actual Pos     4133    863
Actual Neg      902   4102

Positive Class (1): Precision=0.8209, Recall=0.8273, F1=0.8240
Negative Class (0): Precision=0.8262, Recall=0.8197, F1=0.8230

STEP 7: Err

In [80]:
# Question 1: How many positive reviews are misclassified as negative?
print(f"\n1. Positive reviews misclassified as negative: {test_results['fn']}")
print(f"   Negative reviews misclassified as positive: {test_results['fp']}")


1. Positive reviews misclassified as negative: 863
   Negative reviews misclassified as positive: 902


In [81]:
# Question 2: Which type of error is more harmful?
print("\n2. Error Harmfulness Analysis:")
print("-" * 40)
print("""
False Negative (Actual Positive, Predicted Negative):
- The model misses a good movie recommendation
- User might never discover a movie they'd enjoy
- Business impact: Lost engagement, potential revenue
- Severity: HIGH

False Positive (Actual Negative, Predicted Positive):
- The model recommends a bad movie
- User might waste time watching something they dislike
- Business impact: User may lose trust in recommendations
- Severity: MODERATE

In this case, false negatives are more harmful as they represent missed
opportunities for user engagement and potential revenue.
""")


2. Error Harmfulness Analysis:
----------------------------------------

False Negative (Actual Positive, Predicted Negative):
- The model misses a good movie recommendation
- User might never discover a movie they'd enjoy
- Business impact: Lost engagement, potential revenue
- Severity: HIGH

False Positive (Actual Negative, Predicted Positive):
- The model recommends a bad movie
- User might waste time watching something they dislike
- Business impact: User may lose trust in recommendations
- Severity: MODERATE

In this case, false negatives are more harmful as they represent missed
opportunities for user engagement and potential revenue.



In [82]:
# ============================================
# STEP 8: INTERPRETATION - Top Words
# ============================================
print("\n" + "="*80)
print("STEP 8: Model Interpretation - Top Words")
print("="*80)


STEP 8: Model Interpretation - Top Words


In [83]:
# Get top positive and negative words
word_weights = [(index_to_word[i], final_model.weights[i]) for i in range(len(vocab))]
word_weights.sort(key=lambda x: x[1], reverse=True)

In [84]:
print("\nTop 10 Most Positive Words (push toward positive sentiment):")
print("-" * 40)
for word, weight in word_weights[:10]:
    print(f"  {word}: {weight:.4f}")


Top 10 Most Positive Words (push toward positive sentiment):
----------------------------------------
  great: 0.5566
  love: 0.3535
  best: 0.3272
  excellent: 0.2876
  wonderful: 0.2648
  well: 0.2514
  enjoy: 0.2189
  beautiful: 0.2185
  perfect: 0.2143
  lov: 0.2136


In [85]:
print("\nTop 10 Most Negative Words (push toward negative sentiment):")
print("-" * 40)
for word, weight in word_weights[-10:][::-1]:
    print(f"  {word}: {weight:.4f}")


Top 10 Most Negative Words (push toward negative sentiment):
----------------------------------------
  bad: -0.7856
  not: -0.4832
  worst: -0.4552
  no: -0.3535
  movie: -0.3386
  waste: -0.3196
  awful: -0.3033
  poor: -0.2971
  even: -0.2920
  bor: -0.2878


In [87]:
print("\nDiscussion:")
print("-" * 40)
print("""
The weights show which words strongly influence the model's decisions:
- Positive words like 'excellent', 'amazing', 'wonderful' push toward positive sentiment
- Negative words like 'terrible', 'awful', 'boring' push toward negative sentiment

Note that negation words like 'not' appear in both lists depending on context,
which highlights a limitation of bag-of-words models - they can't capture
phrases like 'not good' where the meaning is opposite of individual words.
""")


Discussion:
----------------------------------------

The weights show which words strongly influence the model's decisions:
- Positive words like 'excellent', 'amazing', 'wonderful' push toward positive sentiment
- Negative words like 'terrible', 'awful', 'boring' push toward negative sentiment

Note that negation words like 'not' appear in both lists depending on context,
which highlights a limitation of bag-of-words models - they can't capture
phrases like 'not good' where the meaning is opposite of individual words.



In [88]:
# ============================================
# STEP 9: MISCLASSIFIED EXAMPLES ANALYSIS
# ============================================
print("\n" + "="*80)
print("STEP 9: Misclassified Examples Analysis")
print("="*80)


STEP 9: Misclassified Examples Analysis


In [89]:
# Get misclassified indices
y_pred_test = final_model.predict(X_test)
misclassified_idx = np.where(y_pred_test != y_test)[0]

In [90]:
print(f"\nTotal misclassified reviews: {len(misclassified_idx)} out of {len(y_test)}")
print(f"Misclassification rate: {len(misclassified_idx)/len(y_test)*100:.2f}%")

print("\nExample Misclassified Reviews:")
print("-" * 60)

for i, idx in enumerate(misclassified_idx[:5]):
    actual = "POSITIVE" if y_test[idx] == 1 else "NEGATIVE"
    predicted = "POSITIVE" if y_pred_test[idx] == 1 else "NEGATIVE"
    
    print(f"\nExample {i+1} (Index: {idx}):")
    print(f"Actual: {actual} | Predicted: {predicted}")
    print(f"Review excerpt: {test_df['review'].iloc[idx][:200]}...")
    
    # Show key words that influenced prediction
    review_tokens = test_df['tokens'].iloc[idx]
    word_influences = []
    
    for word in set(review_tokens):
        if word in vocab:
            word_idx = vocab[word]
            # Influence = presence * weight
            influence = final_model.weights[word_idx]
            word_influences.append((word, influence))
    
    word_influences.sort(key=lambda x: abs(x[1]), reverse=True)
    
    print("Top influencing words:")
    for word, influence in word_influences[:5]:
        direction = "positive" if influence > 0 else "negative"
        print(f"  '{word}': {influence:.4f} ({direction})")
    
    # Check for negation patterns
    negation_words = {'not', 'no', 'never', 'nothing'}
    has_negation = any(word in negation_words for word in review_tokens)
    if has_negation:
        print("  Note: Contains negation words - may confuse bag-of-words model")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"""
Model Performance Summary:
-------------------------
Training Accuracy: {train_results['accuracy']:.2%}
Validation Accuracy: {val_results['accuracy']:.2%}
Test Accuracy: {test_results['accuracy']:.2%}

Key Findings:
1. The model achieves reasonable accuracy considering no ML libraries were used
2. Most informative positive words: {', '.join([w for w,_ in word_weights[:5]])}
3. Most informative negative words: {', '.join([w for w,_ in word_weights[-5:]])}
4. Common error cases involve negation and complex sentence structures
5. False negatives (missed positives) are more harmful than false positives

Limitations of Bag-of-Words Approach:
- Cannot capture word order ("not good" vs "good not")
- Loses context and syntax
- Treats each word independently
- Negation handling is limited
""")


Total misclassified reviews: 1765 out of 10000
Misclassification rate: 17.65%

Example Misclassified Reviews:
------------------------------------------------------------

Example 1 (Index: 5):
Actual: POSITIVE | Predicted: NEGATIVE
Review excerpt: I've seen a fair few films from the Far East recently.....some were excellent (Battle Royale, Infernal Affairs, the Eye), and some were not so great (Versus, The Triple Cross). Then there are ones lik...
Top influencing words:
  'great': 0.5566 (positive)
  'not': -0.4832 (negative)
  'bor': -0.2878 (negative)
  'excellent': 0.2876 (positive)
  'could': -0.2341 (negative)
  Note: Contains negation words - may confuse bag-of-words model

Example 2 (Index: 9):
Actual: NEGATIVE | Predicted: POSITIVE
Review excerpt: I gave this 3 stars out of a possible 10 - because the stories are open-ended and left unexplained, and because of the nauseating scenes of someone eating in an extremely disgusting way, plus scenes o...
Top influencing words:
  'no